# LiveSQLBench 角色 SQL 数据集构建（SELECT-only）

本笔记本演示如何基于 LiveSQLBench 数据生成带角色约束的 Text-to-SQL 数据集，当前暂时仅支持 SELECT 语句，以便与角色权限模型对齐。

## 使用说明

- 建议先运行 `pip install -r requirements.txt` 并配置好 OpenAI/DeepSeek 等模型的 API Key。
- LiveSQLBench 数据应解压到 `data/livesqlbench-base-lite-sqlite`（或相应版本）目录。
- 默认会尝试复用 `outputs/livesql_data/` 下最近一次生成的角色分配结果，只有在没有缓存时才会触发 LLM 生成流程。
- 当前仅导出 SELECT 语句，以方便在权限模型扩展前保持一致性。

In [1]:
import json
import logging
import os
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not PROJECT_ROOT or not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Project root not found. Please run this notebook inside the repository.")

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
LIVESQL_OUTPUT_DIR = OUTPUT_ROOT / "livesql_data"
LIVESQL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.role_parser import ParallelRoleGenerator
from src.processors.livesql_describer import LiveSQLDatabaseDescriber
from src.processors.livesql_role_processor import LiveSQLRoleProcessor
from src.processors.livesql_role_sql_generator import LiveSQLRoleSQLGenerator

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("livesql_notebook")

logger.info("Project root: %s", PROJECT_ROOT)

/home/feiy/anaconda3/envs/llm4db/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO | livesql_notebook | Project root: /home/feiy/Role-SQL-benchmark


In [2]:
def find_latest_file(directory: Path, prefix: str, suffix: str = ".json") -> Optional[Path]:
    """Locate the most recent file with the given prefix in ``directory``."""
    candidates = list(directory.glob(f"{prefix}*{suffix}"))
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


def load_cached_role_assignments(role_path: Optional[Path] = None) -> Optional[Dict[str, Sequence[Dict[str, str]]]]:
    """Load cached role assignments from disk if available."""
    if role_path is None:
        role_path = find_latest_file(LIVESQL_OUTPUT_DIR, "role_assignments_")
    if role_path is None or not role_path.exists():
        logger.warning("Cached role assignments not found. Configure API credentials to regenerate roles.")
        return None
    logger.info("Reusing cached roles: %s", role_path)
    with open(role_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    assignments = data.get("assignments")
    if not assignments:
        logger.error("Role cache has unexpected format: %s", role_path)
        return None
    return assignments


def timestamped_filename(prefix: str, extension: str = "json") -> Path:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return LIVESQL_OUTPUT_DIR / f"{prefix}_{stamp}.{extension}"

In [3]:
def prepare_role_assignments(
    *,
    describer: LiveSQLDatabaseDescriber,
    db_names: Optional[Iterable[str]] = None,
    reuse_cached: bool = True,
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    n_workers: int = 4,
 ) -> Tuple[Path, Dict[str, Sequence[Dict[str, str]]]]:
    """Ensure role assignments exist and return ``(path, assignments)``."""
    if reuse_cached:
        cached_path = find_latest_file(LIVESQL_OUTPUT_DIR, "role_assignments_")
        if cached_path:
            logger.info("Attempting to reuse role cache: %s", cached_path)
            assignments = load_cached_role_assignments(cached_path)
            if assignments:
                return cached_path, assignments
            logger.warning("Cached role file failed to load; generating roles from scratch.")

    model = model or os.getenv("LIVESQL_ROLE_MODEL", os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
    api_key = api_key or os.getenv("LIVESQL_ROLE_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("API key not found. Set OPENAI_API_KEY or LIVESQL_ROLE_API_KEY to regenerate roles.")

    logger.info("Regenerating role assignments with model=%s", model)
    generator = ParallelRoleGenerator(model=model, api_key=api_key, n_workers=n_workers)
    role_processor = LiveSQLRoleProcessor(generator, describer)
    result = role_processor.process_databases(db_names=db_names)
    assignments = result.get("assignments", {})
    if not assignments:
        raise RuntimeError("Role generation returned no assignments. Inspect model output for issues.")

    saved_path = role_processor.save_role_assignments(result, LIVESQL_OUTPUT_DIR)
    return saved_path, assignments


def run_livesql_pipeline(
    *,
    db_names: Optional[Iterable[str]] = None,
    reuse_cached_roles: bool = True,
    allowed_statement_types: Optional[Iterable[str]] = ("select",),
    save_dataset: bool = True,
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    n_workers: int = 4,
 ) -> Dict[str, object]:
    """Execute the LiveSQL pipeline end-to-end and return artefacts."""
    describer = LiveSQLDatabaseDescriber(PROJECT_ROOT)
    role_path, assignments = prepare_role_assignments(
        describer=describer,
        db_names=db_names,
        reuse_cached=reuse_cached_roles,
        model=model,
        api_key=api_key,
        n_workers=n_workers,
    )

    sql_generator = LiveSQLRoleSQLGenerator(
        PROJECT_ROOT,
        describer=describer,
        output_dir=str(LIVESQL_OUTPUT_DIR),
    )

    normalized_allowed = None if allowed_statement_types is None else {typ.lower() for typ in allowed_statement_types}
    dataset = sql_generator.generate_livesql_role_sql_dataset(
        role_file_path=str(role_path),
        data_source="dev",
        allowed_statement_types=normalized_allowed,
    )

    dataset_path = None
    if dataset and save_dataset:
        dataset_path = sql_generator.save_livesql_dataset(dataset)

    return {
        "dataset": dataset,
        "dataset_path": dataset_path,
        "assignments_path": role_path,
        "assignments": assignments,
        "generator": sql_generator,
    }

### Baseline 数据转换工具

In [4]:
def build_livesql_baseline_dataset(role_dataset: Sequence[Dict[str, object]]) -> Tuple[list, Dict[str, int]]:
    """Convert role-aware LiveSQL examples into a baseline dataset without role constraints."""
    diagnostics = {
        "total_rows": len(role_dataset),
        "missing_instance_id": 0,
        "duplicate_instances": 0,
        "missing_gold_sql": 0,
    }

    baseline_map: Dict[str, Dict[str, object]] = {}

    for entry in role_dataset:
        metadata = entry.get("metadata") or {}
        instance_id = metadata.get("instance_id")

        if not instance_id:
            diagnostics["missing_instance_id"] += 1
            continue

        if instance_id in baseline_map:
            diagnostics["duplicate_instances"] += 1
            continue

        gold_sql = metadata.get("gold_sql") or entry.get("output") or ""
        if isinstance(gold_sql, list):
            gold_sql = "\n".join(gold_sql)

        if not gold_sql:
            diagnostics["missing_gold_sql"] += 1
            continue

        baseline_map[instance_id] = {
            "db_id": entry.get("db_id"),
            "instruction": entry.get("instruction"),
            "input": entry.get("input"),
            "output": gold_sql,
        }

    baseline_dataset = list(baseline_map.values())
    diagnostics["unique_instances"] = len(baseline_dataset)
    return baseline_dataset, diagnostics

## 参数配置

在执行管线前可以根据需要调整数据库筛选、允许的语句类型、是否复用缓存角色以及是否保存生成的数据集。

In [5]:
# Configure LiveSQL pipeline parameters before running
PIPELINE_CONFIG = {
    "db_names": None,  # Limit to a subset like ("virtual", "geo"), None means all databases
    "reuse_cached_roles": True,  # Prefer cached role assignments when available
    "allowed_statement_types": ("select",),  # Keep only the specified SQL statement types
    "save_dataset": True,  # Persist generated datasets under outputs/livesql_data
    "model": None,  # Optional override for the role generation model
    "api_key": None,  # Optional override for API key; falls back to environment variables
    "n_workers": 4,  # Parallel workers when invoking the LLM role generator
}
PIPELINE_CONFIG

{'db_names': None,
 'reuse_cached_roles': True,
 'allowed_statement_types': ('select',),
 'save_dataset': True,
 'model': None,
 'api_key': None,
 'n_workers': 4}

### 执行管线

In [6]:
artifacts = run_livesql_pipeline(**PIPELINE_CONFIG)
dataset = artifacts.get("dataset", [])
dataset_path = artifacts.get("dataset_path")
assignments_path = artifacts.get("assignments_path")
sql_generator = artifacts.get("generator")
assignments_path_display = str(assignments_path) if assignments_path else "Not generated (cache reused)"
dataset_path_display = str(dataset_path) if dataset_path else "Not saved"
summary_lines = [
    "- Dataset size: **{}**".format(len(dataset)),
    "- Role assignment file: `{}`".format(assignments_path_display),
    "- Exported dataset file: `{}`".format(dataset_path_display),
    "- Total examples observed: {}".format(getattr(sql_generator, "get_last_total_example_count", lambda: "unknown")()),
    "- Examples filtered by statement type: {}".format(getattr(sql_generator, "get_last_filtered_example_count", lambda: "unknown")()),
]
display(Markdown("\n".join(["### Run summary"] + summary_lines)))

INFO | livesql_notebook | Attempting to reuse role cache: /home/feiy/Role-SQL-benchmark/outputs/livesql_data/role_assignments_20251001_142136_livesql_dev.json
INFO | livesql_notebook | Reusing cached roles: /home/feiy/Role-SQL-benchmark/outputs/livesql_data/role_assignments_20251001_142136_livesql_dev.json
INFO | livesql_role_sql | Loaded 270 LiveSQL examples from /home/feiy/Role-SQL-benchmark/data/livesqlbench-base-lite-sqlite/livesqlbench_data_sqlite.jsonl
INFO | livesql_role_sql | Loaded ground-truth metadata for 270 LiveSQL examples
INFO | livesql_notebook | Reusing cached roles: /home/feiy/Role-SQL-benchmark/outputs/livesql_data/role_assignments_20251001_142136_livesql_dev.json
INFO | livesql_role_sql | Loaded 270 LiveSQL examples from /home/feiy/Role-SQL-benchmark/data/livesqlbench-base-lite-sqlite/livesqlbench_data_sqlite.jsonl
INFO | livesql_role_sql | Loaded ground-truth metadata for 270 LiveSQL examples
WARNING | role_sql | No tables extracted from query: SELECT (10.0 - ABS(2

### Run summary
- Dataset size: **872**
- Role assignment file: `/home/feiy/Role-SQL-benchmark/outputs/livesql_data/role_assignments_20251001_142136_livesql_dev.json`
- Exported dataset file: `/home/feiy/Role-SQL-benchmark/outputs/livesql_data/role_sql_dataset_livesql_20251001_153817_dev.json`
- Total examples observed: 270
- Examples filtered by statement type: 92

### 基线数据集导出（无角色信息）

In [7]:
baseline_dataset = []
baseline_dataset_path: Optional[str] = None
baseline_diagnostics: Dict[str, int] = {}

if dataset:
    baseline_dataset, baseline_diagnostics = build_livesql_baseline_dataset(dataset)

    if baseline_dataset:
        baseline_path = timestamped_filename("baseline_sql_dataset_livesql_dev")
        with open(baseline_path, "w", encoding="utf-8") as f:
            json.dump(baseline_dataset, f, ensure_ascii=False, indent=2)
        baseline_dataset_path = str(baseline_path)

        summary_lines = [
            "#### Baseline summary",
            f"- Unique examples: **{baseline_diagnostics.get('unique_instances', len(baseline_dataset))}**",
            f"- Exported file: `{baseline_dataset_path}`",
            f"- Total role rows processed: {baseline_diagnostics.get('total_rows', len(dataset))}",
            f"- Rows skipped (missing instance_id): {baseline_diagnostics.get('missing_instance_id', 0)}",
            f"- Rows skipped (duplicates): {baseline_diagnostics.get('duplicate_instances', 0)}",
            f"- Rows skipped (missing gold SQL): {baseline_diagnostics.get('missing_gold_sql', 0)}",
        ]
        display(Markdown("\n".join(summary_lines)))
    else:
        logger.warning("Baseline dataset is empty after conversion. Diagnostics: %s", baseline_diagnostics)
else:
    logger.warning("Role-aware dataset empty; skipping baseline export.")

#### Baseline summary
- Unique examples: **178**
- Exported file: `/home/feiy/Role-SQL-benchmark/outputs/livesql_data/baseline_sql_dataset_livesql_dev_20251001_153817.json`
- Total role rows processed: 872
- Rows skipped (missing instance_id): 0
- Rows skipped (duplicates): 694
- Rows skipped (missing gold SQL): 0

### Baseline 数据样例与字段校验

In [8]:
if baseline_dataset:
    baseline_df = pd.DataFrame(baseline_dataset)
    expected_keys = {"db_id", "instruction", "input", "output"}
    observed_keys = set(baseline_df.columns)
    missing_keys = expected_keys - observed_keys
    extra_keys = observed_keys - expected_keys

    preview = baseline_df.head(3)
    summary_lines = [
        "#### Baseline dataset check",
        f"- Columns observed: {sorted(observed_keys)}",
        f"- Missing expected columns: {sorted(missing_keys) if missing_keys else 'None'}",
        f"- Extra columns: {sorted(extra_keys) if extra_keys else 'None'}",
        f"- Preview rows: {len(preview)} / {len(baseline_df)}",
    ]
    display(Markdown("\n".join(summary_lines)))
    display(preview)
else:
    baseline_df = pd.DataFrame()
    logger.info("Baseline dataset is empty; skip preview.")

#### Baseline dataset check
- Columns observed: ['db_id', 'input', 'instruction', 'output']
- Missing expected columns: None
- Extra columns: None
- Preview rows: 3 / 178

,db_id,instruction,input,output
0,alien,"# Database Schema:\nCREATE TABLE ""observationa...",I want to analyze how the Signal-to-Noise Qual...,WITH signal_quality AS ( SELECT s.SignalRegist...
1,alien,"# Database Schema:\nCREATE TABLE ""observationa...",I want to find signals that might contain stru...,WITH stability_analysis AS ( SELECT s.SignalRe...
2,alien,"# Database Schema:\nCREATE TABLE ""observationa...","Classify signals by TOLS Category, and for eac...",SELECT CASE WHEN p.TechSigProb * (1 - p.NatSrc...


### 语句类型与权限统计

In [9]:
statement_counts: Dict[str, int] = {}
if sql_generator is not None:
    statement_counts = sql_generator.get_last_statement_type_counts()
allowed_count = sum(1 for entry in dataset if entry.get("output") != "Sorry, I cannot answer.")
denied_count = len(dataset) - allowed_count
stats_lines = [
    "- Allowed examples: **{}**".format(allowed_count),
    "- Permission-denied examples: **{}**".format(denied_count),
    "- Statement type histogram: `{}`".format(statement_counts or "{}"),
]
display(Markdown("\n".join(["### Permission stats"] + stats_lines)))

### Permission stats
- Allowed examples: **310**
- Permission-denied examples: **562**
- Statement type histogram: `{'select': 178, 'write': 64, 'ddl': 27, 'unknown': 1}`

### 数据集分析

In [10]:
if dataset:
    dataset_df = pd.json_normalize(dataset)
    dataset_df["allowed"] = dataset_df["output"].ne("Sorry, I cannot answer.")
    display(dataset_df.head())
else:
    dataset_df = pd.DataFrame()
    logger.warning("Dataset is empty; analysis outputs are skipped.")

,db_id,instruction,role,tables,input,output,difficulty,metadata.instance_id,metadata.gold_sql,metadata.raw_query_tables,metadata.query_tables,metadata.difficulty,metadata.category,metadata.difficulty_tier,metadata.role_description,metadata.high_level,metadata.external_knowledge,metadata.test_cases,allowed
0,alien,"# Database Schema:\nCREATE TABLE ""observationa...",SystemManager,"observationalconditions, observatories, resear...",I want to analyze how the Signal-to-Noise Qual...,WITH signal_quality AS ( SELECT s.SignalRegist...,moderate,alien_1,WITH signal_quality AS ( SELECT s.SignalRegist...,"[observatories, ranked_quality, signal_quality...","[observatories, signals, telescopes]",moderate,Query,Moderate,Overall management and access to all system ta...,True,"[0, 50]",NaN,True
1,alien,"# Database Schema:\nCREATE TABLE ""observationa...",ObservatoryManager,"observatories, telescopes, observationalcondit...",I want to analyze how the Signal-to-Noise Qual...,"Sorry, I cannot answer.",moderate,alien_1,WITH signal_quality AS ( SELECT s.SignalRegist...,"[observatories, ranked_quality, signal_quality...","[observatories, signals, telescopes]",moderate,Query,Moderate,Responsible for managing observatory-related d...,True,"[0, 50]",NaN,False
2,alien,"# Database Schema:\nCREATE TABLE ""observationa...",ResearchAnalyst,"researchprocess, signaladvancedphenomena, sign...",I want to analyze how the Signal-to-Noise Qual...,"Sorry, I cannot answer.",moderate,alien_1,WITH signal_quality AS ( SELECT s.SignalRegist...,"[observatories, ranked_quality, signal_quality...","[observatories, signals, telescopes]",moderate,Query,Moderate,Focused on analyzing research processes and si...,True,"[0, 50]",NaN,False
3,alien,"# Database Schema:\nCREATE TABLE ""observationa...",SignalTechnician,"signaldecoding, signaldynamics, signalprobabil...",I want to analyze how the Signal-to-Noise Qual...,"Sorry, I cannot answer.",moderate,alien_1,WITH signal_quality AS ( SELECT s.SignalRegist...,"[observatories, ranked_quality, signal_quality...","[observatories, signals, telescopes]",moderate,Query,Moderate,Handles technical aspects of signal processing...,True,"[0, 50]",NaN,False
4,alien,"# Database Schema:\nCREATE TABLE ""observationa...",SourceResearcher,"sourceproperties, signals",I want to analyze how the Signal-to-Noise Qual...,"Sorry, I cannot answer.",moderate,alien_1,WITH signal_quality AS ( SELECT s.SignalRegist...,"[observatories, ranked_quality, signal_quality...","[observatories, signals, telescopes]",moderate,Query,Moderate,Conducts research on astronomical sources and ...,True,"[0, 50]",NaN,False


In [11]:
if not dataset_df.empty:
    role_stats = dataset_df.groupby("role").size().rename("count").sort_values(ascending=False)
    display(Markdown("#### Top 10 roles by coverage"))
    display(role_stats.head(10).to_frame())
    db_stats = dataset_df.groupby("db_id").size().rename("count").sort_values(ascending=False)
    display(Markdown("#### Top 10 databases by coverage"))
    display(db_stats.head(10).to_frame())
    allowance_ratio = dataset_df["allowed"].mean()
    display(Markdown(f"#### Permission pass rate: {allowance_ratio:.2%}"))
else:
    logger.info("No dataset rows available for aggregate statistics.")

#### Top 10 roles by coverage

,count
role,
SystemManager,178
DataAnalyst,39
ComplianceOfficer,39
MaintenanceTechnician,30
VendorManager,21
EnvironmentalMonitor,21
FinancialAnalyst,20
DeviceManager,18
SafetyInspector,12


#### Top 10 databases by coverage

,count
db_id,
polar,60
cross_db,60
virtual,60
archeology,55
cybermarket,55
solar,54
alien,50
gaming,50
fake,50


#### Permission pass rate: 35.55%

In [12]:
if not dataset_df.empty:
    sample = dataset_df.sample(n=min(3, len(dataset_df)), random_state=42)
    previews = []
    for _, row in sample.iterrows():
        preview_lines = [
            f"**Database**: `{row['db_id']}`",
            f"**Role**: `{row['role']}`",
            f"**Permitted**: {'✅' if row['allowed'] else '⛔️'}",
            "",
            "**User query:**",
            row.get("input", ""),
            "",
            "**Model response / gold SQL:**",
            row.get("output", "")[:2000],
        ]
        previews.append("\n".join(preview_lines))
    display(Markdown("\n\n---\n\n".join(previews)))

**Database**: `vaccine`
**Role**: `MaintenanceOfficer`
**Permitted**: ⛔️

**User query:**
For each container, I want to identify those with a Temperature Alert. Please list the container ID, Temperature Breach Severity, number of temperature deviations, and an array of alert types for each sensor reading. Sort by TBS from highest to lowest and limit to 5 results.

**Model response / gold SQL:**
Sorry, I cannot answer.

---

**Database**: `disaster`
**Role**: `SystemManager`
**Permitted**: ✅

**User query:**
List the Environmental Impact Factor of each disaster operation by showing the environment health registry, disaster registry, affected area, hazard type, calculated Environmental Impact Factor (rounded to 2 decimal places), and its corresponding Environmental Impact Classification. Sort results from lowest to highest EIF.

**Model response / gold SQL:**
SELECT e.envhealthregistry, d.distregistry, d.affectedarea, d.haztype, ROUND( e.carbontons * ( 1 - e.renewenergypct / 100.0 ) + ( 100 - e.recyclepct ) * 0.5, 2 ) AS environmental_impact_factor, CASE WHEN e.carbontons * ( 1 - e.renewenergypct / 100.0 ) + ( 100 - e.recyclepct ) * 0.5 < 50 THEN 'Sustainable' WHEN e.carbontons * ( 1 - e.renewenergypct / 100.0 ) + ( 100 - e.recyclepct ) * 0.5 < 100 THEN 'Moderate Impact' ELSE 'High Impact' END AS sustainability_assessment FROM environmentandhealth AS e JOIN disasterevents AS d ON e.envdistref = d.distregistry ORDER BY environmental_impact_factor NULLS LAST

---

**Database**: `fake`
**Role**: `ModerationOfficer`
**Permitted**: ⛔️

**User query:**
Find the top 10 accounts that could be part of a Behavioral Anomaly Cluster. For each account, show their account ID, Behavioral Anomaly Score value, and Latest Bot Likelihood Score. Only include accounts with bot likelihood scores above 70. Sort results by BAS in descending order.

**Model response / gold SQL:**
Sorry, I cannot answer.

### Prompt 摘要预览
展示部分样本的完整指令与角色/SQL 摘要，方便比对官方模板和当前实现是否一致。

In [13]:
if not dataset_df.empty:
    example_row = dataset_df.iloc[0]
    full_instruction = example_row['instruction']
    role_description = example_row.get('metadata.role_description', 'N/A')
    gold_sql = example_row.get('metadata.gold_sql', 'N/A')
    external_knowledge = example_row.get('metadata.external_knowledge', 'N/A')
    display(Markdown("#### Full instruction"))
    display(Markdown("````\n" + full_instruction + "\n````"))
    display(Markdown("#### Role description"))
    display(Markdown(role_description if role_description != 'N/A' else "_Not provided_"))
    display(Markdown("#### Gold SQL"))
    display(Markdown("````\n" + str(gold_sql) + "\n````"))
    if external_knowledge != 'N/A':
        display(Markdown("#### External knowledge"))
        display(Markdown("````\n" + str(external_knowledge) + "\n````"))
else:
    logger.info("Dataset is empty; skipping instruction preview.")

#### Full instruction

````
# Database Schema:
CREATE TABLE "observationalconditions" (
signalref text(36) NOT NULL,
obstime text(6) NULL,
obsdate TEXT NULL,
obsdurhrs real(5,2) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref    obstime    obsdate       obsdurhrs
AS796980     00:00:00   2018-11-27        22.04
AS745021     00:00:00   2024-09-02         2.35
AS499555     00:00:00   2023-03-15         6.88
...

CREATE TABLE "observatories" (
observstation text(60) NOT NULL,
weathprofile text(40) NULL,
seeingprofile text(50) NULL,
atmostransparency real(5,3) NULL,
lunarstage text(25) NULL,
lunardistdeg real(7,2) NULL,
solarstatus text(35) NULL,
geomagstatus text(35) NULL,
sidereallocal text(8) NULL,
airtempc real(5,2) NULL,
humidityrate real(6,3) NULL,
windspeedms real(4,2) NULL,
presshpa real(6,1) NULL,
    PRIMARY KEY (observstation)
);
First 3 rows:
observstation                weathprofile    seeingprofile      atmostransparency  lunarstage       lunardistdeg  solarstatus    geomagstatus      sidereallocal    airtempc    humidityrate    windspeedms    presshpa
Observatory-East Darrenport  Clear           Good                            0.04  First Quarter          125.94  High           Quiet                   17.2762        37.6            21.5           26.7      1028
Observatory-Pearsonstad      Clear           Poor                            0.25  Full                   100.56  Low            Quiet                   17.5804       -16.9            67.7           22        1022.3
Observatory-New Lindastad    Cloudy          Good                            0.21  Last Quarter            98.09  High           Storm                   10.9481       -12.5            25.7            4.4      1011
...

CREATE TABLE "researchprocess" (
signalref text(36) NOT NULL,
analysisprio TEXT NULL,
followstat text(25) NULL,
peerrevstat text(25) NULL,
pubstat text(25) NULL,
resprio text(30) NULL,
fundstat text(30) NULL,
collabstat text(35) NULL,
secclass text(35) NULL,
discstat text(40) NULL,
notesmemo TEXT NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref    analysisprio    followstat    peerrevstat    pubstat    resprio    fundstat    collabstat     secclass    discstat    notesmemo
AS796980     Low             Completed     Completed      Submitted  Medium     Unfunded    International  Classified
AS745021     Medium          Completed     In Progress    Published  Low        Pending     Team           Public      Full
AS499555     Urgent          Scheduled     Completed      Published  High       Pending     Solo           Classified              While why recognize what probably sport.
...

CREATE TABLE "signaladvancedphenomena" (
signalref text(36) NOT NULL,
intermedeffects text(40) NULL,
gravlens text(50) NULL,
quanteffects text(85) NULL,
encryptevid text(40) NULL,
langstruct TEXT NULL,
msgcontent TEXT NULL,
cultsig text(60) NULL,
sciimpact text(50) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref    intermedeffects    gravlens    quanteffects    encryptevid    langstruct    msgcontent    cultsig    sciimpact
AS796980     Severe             Weak        Significant     Strong         Complex       Identified    High       Major
AS745021     Minimal                        Significant     Strong         Simple                                 Major
AS499555     Minimal            Weak                        Strong         Simple        Possible                 Major
...

CREATE TABLE "signalclassification" (
signalref text(36) NOT NULL,
sigclasstype text(40) NULL,
sigpattern text(60) NULL,
repeatcount integer(16) NULL,
periodsec real(12,3) NULL,
complexidx real(6,3) NULL,
entropyval real(6,2) NULL,
infodense real(6,3) NULL,
classconf real(5,2) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref    sigclasstype    sigpattern      repeatcount    periodsec    complexidx    entropyval    infodense    classconf
AS796980     Artificial      Unknown                  96     10452.6          0.161          4.08        0.849         71.3
AS745021     Unknown         Unknown                   1     11646.1          0.189          4.44        0.922         86.4
AS499555     Artificial      Random                   32      8643.14         0.915          2.71        0.688          9.4
...

CREATE TABLE "signaldecoding" (
signalref text(36) NOT NULL,
encodetype text(40) NULL,
compressratio real(6,3) NULL,
errcorrlvl text(35) NULL,
decodeconf real(5,2) NULL,
decodemethod text(35) NULL,
decodestat text(25) NULL,
decodeiters integer(16) NULL,
proctimehrs real(6,2) NULL,
compresources text(50) NULL,
analysisdp text(25) NULL,
veriflvl text(30) NULL,
confirmstat text(30) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref    encodetype      compressratio  errcorrlvl      decodeconf  decodemethod    decodestat      decodeiters    proctimehrs  compresources    analysisdp     veriflvl    confirmstat
AS796980     Unknown                  5.03  Medium                98.6  Wavelet         Completed               103         193.8   High             Comprehensive  Partially   Pending
AS745021     Unknown                  1.28  Medium                76.3  FFT             In Progress             684         276.22  Low              Detailed       Partially   Confirmed
AS499555     Unknown                  9.2   Low                   25.6  FFT             Completed               486         796.01  High             Comprehensive  Unverified  Pending
...

CREATE TABLE "signaldynamics" (
signalref text(36) NOT NULL,
sigintegrity text(30) NULL,
sigrecurr text(25) NULL,
sigevolve text(25) NULL,
tempstab text(20) NULL,
spatstab text(20) NULL,
freqstab text(35) NULL,
phasestab text(35) NULL,
ampstab text(20) NULL,
modstab text(30) NULL,
sigcoherence text(25) NULL,
sigdisp text(25) NULL,
sigscint text(45) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref      sigintegrity  sigrecurr    sigevolve      tempstab    spatstab    freqstab    phasestab    ampstab    modstab    sigcoherence    sigdisp    sigscint
AS796980               88.3  Sporadic     Unknown           0.737       0.579       0.039        0.278      0.796      0.062           0.538     227.66       0.854
AS745021               52.3               Static            0.674       0.762       0.673        0.485      0.15       0.241           0.979     514.79       0.122
AS499555               60.5  Regular      Unknown           0.84        0.857       0.775        0.36       0.166      0.332           0.397     582.26       0.449
...

CREATE TABLE "signalprobabilities" (
signalref text(36) NOT NULL,
falseposprob real(5,4) NULL,
sigunique real(7,4) NULL,
simindex real(5,4) NULL,
corrscore real(5,4) NULL,
anomscore integer(53) NULL,
techsigprob real(5,4) NULL,
biosigprob real(6,2) NULL,
natsrcprob real(7,3) NULL,
artsrcprob real(3,1) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref      falseposprob    sigunique    simindex    corrscore    anomscore    techsigprob    biosigprob    natsrcprob    artsrcprob
AS796980              0.033        0.73        0.198        0.937        0.941          0.317          0.88         0.574           0.1
AS745021              0.758        0.744       0.986       -0.109        0.488          0.179          0            0.14            0.9
AS499555              0.467        0.301       0.803        0.333        0.139          0.273          0.33         0.226           0.8
...

CREATE TABLE "signals" (
signalregistry text(36) NOT NULL,
timemark text(6) NULL,
telescref text(20) NOT NULL,
detectinstr text(50) NULL,
signalclass text(50) NULL,
sigstrdb real(7,2) NULL,
freqmhz real(9,3) NULL,
bwhz real(10,3) NULL,
centerfreqmhz real(8,3) NULL,
freqdrifthzs real(9,3) NULL,
doppshifthz integer(53) NULL,
sigdursec real(6,2) NULL,
pulsepersec real(6,3) NULL,
pulsewidms real(6,3) NULL,
modtype text(30) NULL,
modindex real(6,4) NULL,
carrierfreqmhz real(9,3) NULL,
phaseshiftdeg real(6,2) NULL,
polarmode text(30) NULL,
polarangledeg real(5,1) NULL,
snrratio real(6,2) NULL,
noisefloordbm integer(53) NULL,
interflvl text(30) NULL,
rfistat text(30) NULL,
atmointerf text(30) NULL,
    PRIMARY KEY (signalregistry),
    FOREIGN KEY (telescref) REFERENCES telescopes(telescregistry)
);
First 3 rows:
signalregistry    timemark                       telescref    detectinstr        signalclass      sigstrdb    freqmhz    bwhz    centerfreqmhz    freqdrifthzs    doppshifthz    sigdursec    pulsepersec    pulsewidms  modtype      modindex    carrierfreqmhz    phaseshiftdeg  polarmode      polarangledeg    snrratio    noisefloordbm  interflvl    rfistat       atmointerf
AS796980          2021-04-16 13:21:19.864197+08  T4621        Quantum Detector   Narrowband        -186.96    95636.3  401.34          4363.09           0.072         143.65      2660.7           4.286        480.95  FM              0.384           65710.4           268.84  Unknown                127.4       26.19          -140.27  High         Unknown       Severe
AS745021          2022-06-15 16:04:26.868816+08  T3182        Optical Telescope  Broadband         -154.72    43019.4  316.03         82505.9            2.204         777.42      3513.05          9.047        695.24  Unknown         0.532           81576.6           343.61  Linear                  16.4       18.01          -105.55  High         Contaminated  Moderate
AS499555          2019-01-18 07:53:35.868816+08  T6585        Radio Telescope    Modulated         -111.19    28682.2  563.64          6161.35           5.668        -665.39      1627.84          1.072          6.78  Unknown         0.893           52348.2           339.52  Elliptical             119.6       20.82          -128.04  High         Unknown       Moderate
...

CREATE TABLE "sourceproperties" (
signalref text(36) NOT NULL,
sourceradeg real(7,4) NULL,
sourcedecdeg real(7,4) NULL,
sourcedistly real(10,2) NULL,
gallong real(6,2) NULL,
gallat real(6,2) NULL,
celestobj text(75) NULL,
objtype text(50) NULL,
objmag real(5,2) NULL,
objtempk integer(32) NULL,
objmasssol real(6,3) NULL,
objagegyr real(6,3) NULL,
objmetal real(5,3) NULL,
objpropmotion real(7,2) NULL,
objradvel real(7,2) NULL,
    PRIMARY KEY (signalref),
    FOREIGN KEY (signalref) REFERENCES signals(signalregistry)
);
First 3 rows:
signalref      sourceradeg    sourcedecdeg    sourcedistly    gallong    gallat  celestobj    objtype          objmag    objtempk    objmasssol    objagegyr    objmetal    objpropmotion    objradvel
AS796980          154.514          82.6753        546187        55.41     17.71  Star         Main Sequence      9.22        3286        68.74         2.159      -1.31              3.31      -768.04
AS745021          217.798         -40.7914        847077        75.22     58.58  Star         Unknown           18.29       16409        14.613        1.428      -0.391             2.59      -462.82
AS499555            4.4794         85.0668         63130.7      79.1      53.38  Galaxy       Dwarf             -2.02       20113        21.034       10.667      -1.15              1.65      -507.02
...

CREATE TABLE "telescopes" (
telescregistry text(20) NOT NULL,
observstation text(60) NOT NULL,
equipstatus text(35) NULL,
calibrstatus text(50) NULL,
pointaccarc real(6,2) NULL,
trackaccarc real(6,2) NULL,
focusquality text(25) NULL,
detecttempk real(7,2) NULL,
coolsysstatus text(35) NULL,
powerstatus text(30) NULL,
datastorstatus text(35) NULL,
netstatus text(40) NULL,
bandusagepct real(5,2) NULL,
procqueuestatus text(40) NULL,
    PRIMARY KEY (telescregistry),
    FOREIGN KEY (observstation) REFERENCES observatories(observstation)
);
First 3 rows:
telescregistry    observstation                equipstatus    calibrstatus      pointaccarc    trackaccarc  focusquality      detecttempk  coolsysstatus    powerstatus    datastorstatus    netstatus       bandusagepct  procqueuestatus
T4621             Observatory-East Darrenport  Operational    Current                  3.85           5.47  Poor                   227.34  Critical         Main           Available         Limited                 77.2  Full
T3182             Observatory-Pearsonstad      Degraded       Due                      3.96           1.58  Good                    94.66  Warning          Backup         Available         Disconnected             8.5  Empty
T6585             Observatory-New Lindastad    Degraded       Current                  6.93           9.4   Good                   233.2   Normal           Main           Available         Disconnected            48.2  Normal
...

# Column Meanings:
{
  "alien|observatories|observstation": "Full name: 'Observatory Name'. Explanation: This field holds the name or unique identifier for the observatory station. Data type: CHAR(60). Example: 'OBS_STATION_ALPHA'.",
  "alien|observatories|weathprofile": "Full name: 'Weather Profile'. Explanation: A short description of weather conditions during observation. Data type: VARCHAR(40). Possible categories: Clear, Cloudy, Partially Cloudy.",
  "alien|observatories|seeingprofile": "Full name: 'Seeing Profile'. Explanation: Assessment of sky seeing conditions (atmospheric steadiness). Data type: VARCHAR(50). Possible categories: Excellent, Good, Poor.",
  "alien|observatories|atmostransparency": "Full name: 'Atmospheric Transparency'. Explanation: A numeric measure of how transparent the atmosphere is. Data type: NUMERIC(5,3). Example: '0.867'.",
  "alien|observatories|lunarstage": "Full name: 'Lunar Phase'. Explanation: The current phase of the Moon during observation. Data type: VARCHAR(25). Possible categories: First Quarter, Full, Last Quarter, New.",
  "alien|observatories|lunardistdeg": "Full name: 'Moon Distance (Degrees)'. Explanation: Angular distance to the Moon in degrees. Data type: DECIMAL(7,2). Example: '97.52'.",
  "alien|observatories|solarstatus": "Full name: 'Solar Activity'. Explanation: The level of solar activity at the time of observation. Data type: VARCHAR(35). Possible categories: High, Low, Moderate.",
  "alien|observatories|geomagstatus": "Full name: 'Geomagnetic Activity'. Explanation: The level of geomagnetic activity during observation. Data type: VARCHAR(35). Possible categories: Active, Quiet, Storm.",
  "alien|observatories|sidereallocal": "Full name: 'Local Sidereal Time'. Explanation: Sidereal time at the observatory in HH:MM:SS format. Data type: CHAR(8). Example: '12:34:56'.",
  "alien|observatories|airtempc": "Full name: 'Air Temperature (°C)'. Explanation: Ambient temperature in Celsius. Data type: NUMERIC(5,2). Example: '18.45'.",
  "alien|observatories|humidityrate": "Full name: 'Humidity (%)'. Explanation: Relative humidity as a percentage. Data type: NUMERIC(6,3). Example: '62.300'.",
  "alien|observatories|windspeedms": "Full name: 'Wind Speed (m/s)'. Explanation: Wind speed in meters per second. Data type: NUMERIC(4,2). Example: '3.45'.",
  "alien|observatories|presshpa": "Full name: 'Pressure (hPa)'. Explanation: Atmospheric pressure in hectopascals. Data type: DECIMAL(6,1). Example: '1013.2'.",
  "alien|telescopes|telescregistry": "Full name: 'Telescope Registry'. Explanation: A unique identifier for the telescope. Data type: CHAR(20). Example: 'TELESC_0001'.",
  "alien|telescopes|observstation": "Full name: 'Observatory Name Reference'. Explanation: Foreign key linking to the observatory station. Data type: CHAR(60). Example: 'OBS_STATION_ALPHA'.",
  "alien|telescopes|equipstatus": "Full name: 'Equipment Status'. Explanation: Operational state of the telescope. Data type: VARCHAR(35). Possible categories: Degraded, Maintenance, Operational.",
  "alien|telescopes|calibrstatus": "Full name: 'Calibration Status'. Explanation: Status of telescope’s calibration. Data type: VARCHAR(50). Possible categories: Current, Due, Overdue.",
  "alien|telescopes|pointaccarc": "Full name: 'Pointing Accuracy (arcsec)'. Explanation: How precise the telescope can point, in arcseconds. Data type: NUMERIC(6,2). Example: '0.45'.",
  "alien|telescopes|trackaccarc": "Full name: 'Tracking Accuracy (arcsec)'. Explanation: How accurately the telescope can track a target, in arcseconds. Data type: NUMERIC(6,2). Example: '1.20'.",
  "alien|telescopes|focusquality": "Full name: 'Focus Quality'. Explanation: Quality of the telescope’s focusing system. Data type: VARCHAR(25). Possible categories: Excellent, Good, Poor.",
  "alien|telescopes|detecttempk": "Full name: 'Detector Temperature (K)'. Explanation: Temperature of the telescope’s primary detector in Kelvin. Data type: DECIMAL(7,2). Example: '123.45'.",
  "alien|telescopes|coolsysstatus": "Full name: 'Cooling System Status'. Explanation: Status of the telescope’s cooling system. Data type: VARCHAR(35). Possible categories: Critical, Normal, Warning.",
  "alien|telescopes|powerstatus": "Full name: 'Power Supply Status'. Explanation: Indicates which power source or level is in use. Data type: VARCHAR(30). Possible categories: Backup, Critical, Main.",
  "alien|telescopes|datastorstatus": "Full name: 'Data Storage Status'. Explanation: Capacity and status of local data storage. Data type: VARCHAR(35). Possible categories: Available, Critical, Low.",
  "alien|telescopes|netstatus": "Full name: 'Network Status'. Explanation: Status of the network connection for telescope data transfer. Data type: VARCHAR(40). Possible categories: Connected, Disconnected, Limited.",
  "alien|telescopes|bandusagepct": "Full name: 'Bandwidth Usage (%)'. Explanation: Network bandwidth usage as a percentage. Data type: NUMERIC(5,2). Example: '73.25'.",
  "alien|telescopes|procqueuestatus": "Full name: 'Processing Queue Status'. Explanation: Indicates how full the processing queue is. Data type: VARCHAR(40). Possible categories: Empty, Full, Normal.",
  "alien|signals|signalregistry": "Full name: 'Signal Registry'. Explanation: Unique ID for each signal record. Data type: CHAR(36). Example: 'SIG-123E4567-E89B'.",
  "alien|signals|timemark": "Full name: 'Timestamp'. Explanation: The moment of signal detection. In some CSVs, it may be stored as an Excel serial date/time (e.g., '44302.55648' ≈ 2021-05-16 13:20:20 UTC). Data type: TIMESTAMPTZ. Example: '2025-08-01 13:45:00+00'.",
  "alien|signals|telescref": "Full name: 'Telescope Reference'. Explanation: Foreign key to the telescope used for detection. Data type: CHAR(20). Example: 'TELESC_0001'.",
  "alien|signals|detectinstr": "Full name: 'Detection Instrument'. Explanation: The instrument type employed for detecting the signal. Data type: VARCHAR(50). Possible categories: Infrared Array, Optical Telescope, Quantum Detector, Radio Telescope.",
  "alien|signals|signalclass": "Full name: 'Signal Type'. Explanation: Broad classification of the detected signal. Data type: VARCHAR(50). Possible categories: Broadband, Continuous, Modulated, Narrowband, Pulsed.",
  "alien|signals|sigstrdb": "Full name: 'Signal Strength (dB)'. Explanation: Measured strength of the signal in decibels. Data type: NUMERIC(7,2). Example: '12.35'.",
  "alien|signals|freqmhz": "Full name: 'Frequency (MHz)'. Explanation: Signal’s nominal frequency in MHz. Data type: DECIMAL(9,3). Example: '1420.405'.",
  "alien|signals|bwhz": "Full name: 'Bandwidth (Hz)'. Explanation: Signal’s bandwidth in Hertz. Data type: DECIMAL(10,3). Example: '500000.000'.",
  "alien|signals|centerfreqmhz": "Full name: 'Center Frequency (MHz)'. Explanation: Center frequency of the detected signal in MHz. Data type: NUMERIC(8,3). Example: '1420.500'.",
  "alien|signals|freqdrifthzs": "Full name: 'Frequency Drift (Hz/s)'. Explanation: How fast the signal drifts in frequency over time. Data type: NUMERIC(9,3). Example: '0.123'.",
  "alien|signals|doppshifthz": "Full name: 'Doppler Shift (Hz)'. Explanation: Measured Doppler shift of the signal in Hertz. Data type: DOUBLE PRECISION. Example: '142.75'.",
  "alien|signals|sigdursec": "Full name: 'Signal Duration (s)'. Explanation: Total duration of the signal in seconds. Data type: NUMERIC(6,2). Example: '12.50'.",
  "alien|signals|pulsepersec": "Full name: 'Pulse Rate (pulses/sec)'. Explanation: Rate of pulse repetition in the signal. Data type: NUMERIC(6,3). Example: '4.500'.",
  "alien|signals|pulsewidms": "Full name: 'Pulse Width (ms)'. Explanation: Duration of each pulse in milliseconds. Data type: NUMERIC(6,3). Example: '2.300'.",
  "alien|signals|modtype": "Full name: 'Modulation Type'. Explanation: The method of signal modulation. Data type: VARCHAR(30). Possible categories: AM, FM, PM, QAM, Unknown.",
  "alien|signals|modindex": "Full name: 'Modulation Index'. Explanation: Numerical index describing modulation depth. Data type: DECIMAL(6,4). Example: '0.2875'.",
  "alien|signals|carrierfreqmhz": "Full name: 'Carrier Frequency (MHz)'. Explanation: Frequency of the carrier wave in MHz. Data type: DECIMAL(9,3). Example: '1000.000'.",
  "alien|signals|phaseshiftdeg": "Full name: 'Phase Shift (°)'. Explanation: Phase shift in degrees for the signal. Data type: NUMERIC(6,2). Example: '45.00'.",
  "alien|signals|polarmode": "Full name: 'Polarization Mode'. Explanation: Type of signal polarization. Data type: VARCHAR(30). Possible categories: Circular, Elliptical, Linear, Unknown.",
  "alien|signals|polarangledeg": "Full name: 'Polarization Angle (°)'. Explanation: Angle of polarization in degrees. Data type: DECIMAL(5,1). Example: '90.0'.",
  "alien|signals|snrratio": "Full name: 'Signal-to-Noise Ratio'. Explanation: SNR measured for the signal. Data type: DECIMAL(6,2). Example: '18.75'.",
  "alien|signals|noisefloordbm": "Full name: 'Noise Floor (dBm)'. Explanation: Measured noise floor in dBm. Data type: DOUBLE PRECISION. Example: '-85.3'.",
  "alien|signals|interflvl": "Full name: 'Interference Level'. Explanation: Degree of interference around the signal. Data type: VARCHAR(30). Possible categories: High, Low, Medium, None.",
  "alien|signals|rfistat": "Full name: 'RFI Status'. Explanation: Radio Frequency Interference status. Data type: VARCHAR(30). Possible categories: Clean, Contaminated, Unknown.",
  "alien|signals|atmointerf": "Full name: 'Atmospheric Interference'. Explanation: Impact of atmospheric conditions on signal. Data type: VARCHAR(30). Possible categories: Minimal, Moderate, Severe.",
  "alien|signalprobabilities|signalref": "Full name: 'Signal Registry Reference'. Explanation: Primary key referencing the main signal record. Data type: CHAR(36). Example: 'SIG-XYZ-9876'.",
  "alien|signalprobabilities|falseposprob": "Full name: 'False Positive Probability'. Explanation: Probability that the signal is falsely detected. Data type: DECIMAL(5,4). Example: '0.0135'.",
  "alien|signalprobabilities|sigunique": "Full name: 'Signal Uniqueness'. Explanation: A measure of how unique the signal is compared to others. Data type: DECIMAL(7,4). Example: '99.1234'.",
  "alien|signalprobabilities|simindex": "Full name: 'Similarity Index'. Explanation: Compares the signal to known references. Data type: NUMERIC(5,4). Example: '0.8222'.",
  "alien|signalprobabilities|corrscore": "Full name: 'Correlation Score'. Explanation: How well the signal aligns with expected patterns. Data type: DECIMAL(5,4). Example: '0.9950'.",
  "alien|signalprobabilities|anomscore": "Full name: 'Anomaly Score'. Explanation: How unusual or unexpected the signal is. Data type: FLOAT. Example: '1.23'.",
  "alien|signalprobabilities|techsigprob": "Full name: 'Technosignature Probability'. Explanation: Probability that the signal is of artificial technological origin. Data type: DECIMAL(5,4). Example: '0.6789'.",
  "alien|signalprobabilities|biosigprob": "Full name: 'Biosignature Probability'. Explanation: Probability that the signal indicates a biological origin. Data type: DECIMAL(6,2). Example: '45.21'.",
  "alien|signalprobabilities|natsrcprob": "Full name: 'Natural Source Probability'. Explanation: Probability the signal is from a natural source. Data type: NUMERIC(7,3). Example: '98.761'.",
  "alien|signalprobabilities|artsrcprob": "Full name: 'Artificial Source Probability'. Explanation: Probability the signal is from an artificial source. Data type: NUMERIC(3,1). Example: '3.4'.",
  "alien|signaladvancedphenomena|signalref": "Full name: 'Signal Registry Reference'. Explanation: Primary key referencing the main signal. Data type: CHAR(36). Example: 'SIG-ABC-1234'.",
  "alien|signaladvancedphenomena|intermedeffects": "Full name: 'Interstellar Medium Effects'. Explanation: Impact of interstellar medium on the signal. Data type: VARCHAR(40). Possible categories: Minimal, Moderate, Severe.",
  "alien|signaladvancedphenomena|gravlens": "Full name: 'Gravitational Lensing'. Explanation: Whether the signal is affected by gravitational lensing. Data type: VARCHAR(50). Possible categories: None, Strong, Weak.",
  "alien|signaladvancedphenomena|quanteffects": "Full name: 'Quantum Effects'. Explanation: Quantum-level phenomena observed in the signal. Data type: VARCHAR(85). Possible categories: None, Observed, Significant.",
  "alien|signaladvancedphenomena|encryptevid": "Full name: 'Encryption Evidence'. Explanation: Whether there is indication the signal is encrypted. Data type: VARCHAR(40). Possible categories: None, Possible, Strong.",
  "alien|signaladvancedphenomena|langstruct": "Full name: 'Language Structure'. Explanation: Presence of linguistic or structured patterns. Data type: TEXT. Possible categories: Complex, None, Simple.",
  "alien|signaladvancedphenomena|msgcontent": "Full name: 'Message Content'. Explanation: Indicates whether an actual message was detected. Data type: TEXT. Possible categories: Identified, None, Possible.",
  "alien|signaladvancedphenomena|cultsig": "Full name: 'Cultural Significance'. Explanation: The level of cultural impact or interest. Data type: VARCHAR(60). Possible categories: High, Low, None.",
  "alien|signaladvancedphenomena|sciimpact": "Full name: 'Scientific Impact'. Explanation: Significance of the signal to scientific research. Data type: VARCHAR(50). Possible categories: Major, Minor, Moderate.",
  "alien|signalclassification|signalref": "Full name: 'Signal Registry Reference'. Explanation: Foreign key referencing the main signal. Data type: CHAR(36). Example: 'SIG-DEF-5678'.",
  "alien|signalclassification|sigclasstype": "Full name: 'Classification Type'. Explanation: High-level category for the signal. Data type: VARCHAR(40). Possible categories: Artificial, Candidate, Natural, Unknown.",
  "alien|signalclassification|sigpattern": "Full name: 'Signal Pattern'. Explanation: Pattern observed in the signal. Data type: VARCHAR(60). Possible categories: Periodic, Random, Structured, Unknown.",
  "alien|signalclassification|repeatcount": "Full name: 'Repetition Count'. Explanation: Number of times the signal has repeated. Data type: SMALLINT. Example: '3'.",
  "alien|signalclassification|periodsec": "Full name: 'Period (s)'. Explanation: Duration of each cycle if periodic. Data type: NUMERIC(7,3). Example: '12.345'.",
  "alien|signalclassification|complexidx": "Full name: 'Complexity Index'. Explanation: Numeric measure of the signal’s complexity. Data type: DECIMAL(6,3). Example: '5.678'.",
  "alien|signalclassification|entropyval": "Full name: 'Entropy'. Explanation: Entropy measurement of the signal. Data type: DECIMAL(6,2). Example: '3.45'.",
  "alien|signalclassification|infodense": "Full name: 'Information Density'. Explanation: Estimated information per unit time/frequency. Data type: DECIMAL(6,3). Example: '2.345'.",
  "alien|signalclassification|classconf": "Full name: 'Classification Confidence (%)'. Explanation: How confident we are in the assigned signal class. Data type: DECIMAL(5,2). Example: '92.50'.",
  "alien|signaldecoding|signalref": "Full name: 'Signal Registry Reference'. Explanation: Foreign key referencing the main signal entry. Data type: CHAR(36). Example: 'SIG-GHI-9012'.",
  "alien|signaldecoding|encodetype": "Full name: 'Encoding Type'. Explanation: The type of signal encoding used (e.g., Binary). Data type: VARCHAR(40). Possible categories: Binary, Complex, Tertiary, Unknown.",
  "alien|signaldecoding|compressratio": "Full name: 'Compression Ratio'. Explanation: Factor by which the raw signal data was compressed. Data type: DECIMAL(6,3). Example: '2.500'.",
  "alien|signaldecoding|errcorrlvl": "Full name: 'Error Correction Level'. Explanation: Degree of error correction applied. Data type: VARCHAR(35). Possible categories: High, Low, Medium, None.",
  "alien|signaldecoding|decodeconf": "Full name: 'Decoding Confidence (%)'. Explanation: Confidence level that the decoding is correct. Data type: DECIMAL(5,2). Example: '88.75'.",
  "alien|signaldecoding|decodemethod": "Full name: 'Decoding Method'. Explanation: Method used to decode the signal. Data type: VARCHAR(35). Possible categories: FFT, Neural Network, Quantum, Wavelet.",
  "alien|signaldecoding|decodestat": "Full name: 'Decoding Status'. Explanation: Status of the decoding process. Data type: VARCHAR(25). Possible categories: Completed, Failed, In Progress.",
  "alien|signaldecoding|decodeiters": "Full name: 'Decoding Iterations'. Explanation: Number of algorithmic passes attempted. Data type: SMALLINT. Example: '7'.",
  "alien|signaldecoding|proctimehrs": "Full name: 'Processing Time (hours)'. Explanation: Total hours spent decoding. Data type: DECIMAL(6,2). Example: '3.50'.",
  "alien|signaldecoding|compresources": "Full name: 'Computational Resources'. Explanation: Level of computing power used. Data type: VARCHAR(50). Possible categories: Extreme, High, Low, Medium.",
  "alien|signaldecoding|analysisdp": "Full name: 'Analysis Depth'. Explanation: How thoroughly the signal was analyzed. Data type: VARCHAR(25). Possible categories: Comprehensive, Detailed, Preliminary.",
  "alien|signaldecoding|veriflvl": "Full name: 'Verification Level'. Explanation: Extent to which decoding has been verified. Data type: VARCHAR(30). Possible categories: Partially, Unverified, Verified.",
  "alien|signaldecoding|confirmstat": "Full name: 'Confirmation Status'. Explanation: Whether the decoding results have been confirmed. Data type: VARCHAR(30). Possible categories: Confirmed, Pending, Rejected.",
  "alien|signaldynamics|signalref": "Full name: 'Signal Registry Reference'. Explanation: Foreign key referencing the main signal. Data type: CHAR(36). Example: 'SIG-JKL-3456'.",
  "alien|signaldynamics|sigintegrity": "Full name: 'Signal Integrity'. Explanation: Rating of how intact or uncorrupted the signal is. Data type: VARCHAR(30). If no fixed categories, example: 'HighIntegrity'.",
  "alien|signaldynamics|sigrecurr": "Full name: 'Signal Recurrence'. Explanation: Whether the signal recurs over time. Data type: VARCHAR(25). Possible categories: None, Regular, Sporadic.",
  "alien|signaldynamics|sigevolve": "Full name: 'Signal Evolution'. Explanation: Indicates if the signal changes/evolves during observation. Data type: VARCHAR(25). Possible categories: Dynamic, Static, Unknown.",
  "alien|signaldynamics|tempstab": "Full name: 'Temporal Stability'. Explanation: How stable the signal remains over time. Data type: VARCHAR(20). If no fixed categories, example: 'Stable'.",
  "alien|signaldynamics|spatstab": "Full name: 'Spatial Stability'. Explanation: How stable the signal is spatially (e.g., consistent direction). Data type: VARCHAR(20). If no fixed categories, example: 'Moderate'.",
  "alien|signaldynamics|freqstab": "Full name: 'Frequency Stability'. Explanation: How stable the signal’s frequency is. Data type: VARCHAR(35). If no fixed categories, example: 'HighlyStable'.",
  "alien|signaldynamics|phasestab": "Full name: 'Phase Stability'. Explanation: Any variation in signal phase. Data type: VARCHAR(35). If no fixed categories, example: 'VaryingPhase'.",
  "alien|signaldynamics|ampstab": "Full name: 'Amplitude Stability'. Explanation: Consistency of signal amplitude. Data type: VARCHAR(20). If no fixed categories, example: 'Unstable'.",
  "alien|signaldynamics|modstab": "Full name: 'Modulation Stability'. Explanation: Consistency in the signal’s modulation scheme. Data type: VARCHAR(30). If no fixed categories, example: 'Consistent'.",
  "alien|signaldynamics|sigcoherence": "Full name: 'Signal Coherence'. Explanation: How coherent the signal remains over its duration. Data type: VARCHAR(25). If no fixed categories, example: 'HighCoherence'.",
  "alien|signaldynamics|sigdisp": "Full name: 'Signal Dispersion'. Explanation: The degree to which the signal is dispersed (time/frequency smearing). Data type: VARCHAR(25). If no fixed categories, example: 'SignificantDisp'.",
  "alien|signaldynamics|sigscint": "Full name: 'Signal Scintillation'. Explanation: Fluctuation in signal amplitude due to propagation effects. Data type: VARCHAR(45). If no fixed categories, example: 'MildScintillation'.",
  "alien|researchprocess|signalref": "Full name: 'Signal Registry Reference'. Explanation: Foreign key referencing the main signal record. Data type: CHAR(36). Example: 'SIG-MNO-7890'.",
  "alien|researchprocess|analysisprio": "Full name: 'Analysis Priority'. Explanation: Priority assigned for analyzing this signal. Data type: TEXT. Possible categories: High, Low, Medium, Urgent.",
  "alien|researchprocess|followstat": "Full name: 'Follow-up Status'. Explanation: Status of any follow-up observations. Data type: VARCHAR(25). Possible categories: Completed, Required, Scheduled.",
  "alien|researchprocess|peerrevstat": "Full name: 'Peer Review Status'. Explanation: Where this signal stands in peer review. Data type: VARCHAR(25). Possible categories: Completed, In Progress, Pending.",
  "alien|researchprocess|pubstat": "Full name: 'Publication Status'. Explanation: Whether findings about the signal have been published. Data type: CHAR(25). Possible categories: Draft, Published, Submitted.",
  "alien|researchprocess|resprio": "Full name: 'Research Priority'. Explanation: How urgently this signal needs scientific attention. Data type: VARCHAR(30). Possible categories: High, Low, Medium.",
  "alien|researchprocess|fundstat": "Full name: 'Funding Status'. Explanation: Funding situation for further study. Data type: VARCHAR(30). Possible categories: Funded, Pending, Unfunded.",
  "alien|researchprocess|collabstat": "Full name: 'Collaboration Status'. Explanation: The nature of collaboration on this signal. Data type: VARCHAR(35). Possible categories: International, Solo, Team.",
  "alien|researchprocess|secclass": "Full name: 'Security Classification'. Explanation: Visibility and clearance level for the data. Data type: CHAR(35). Possible categories: Classified, Public, Restricted.",
  "alien|researchprocess|discstat": "Full name: 'Disclosure Status'. Explanation: How much information about the signal is shared publicly. Data type: VARCHAR(40). Possible categories: Full, None, Partial.",
  "alien|researchprocess|notesmemo": "Full name: 'Research Notes'. Explanation: Any extra remarks or context by researchers. Data type: TEXT. Example: 'Project requires further funding...'",
  "alien|observationalconditions|signalref": "Full name: 'Signal Registry Reference'. Explanation: Primary key referencing the signal. Data type: CHAR(36). Example: 'SIG-PQR-9876'.",
  "alien|observationalconditions|obstime": "Full name: 'Observation Time'. Explanation: Local time of the observation in HH:MM:SS. Data type: TIME. Example: '13:45:59'.",
  "alien|observationalconditions|obsdate": "Full name: 'Observation Date'. Explanation: Local date of the observation (YYYY-MM-DD). Data type: DATE. Example: '2025-08-01'.",
  "alien|observationalconditions|obsdurhrs": "Full name: 'Observation Duration (hours)'. Explanation: How long the observation lasted in hours. Data type: NUMERIC(5,2). Example: '2.50'.",
  "alien|sourceproperties|signalref": "Full name: 'Signal Registry Reference'. Explanation: Foreign key referencing the main signal. Data type: CHAR(36). Example: 'SIG-STU-5432'.",
  "alien|sourceproperties|sourceradeg": "Full name: 'Right Ascension (°)'. Explanation: RA of the source in degrees (0 to 360). Data type: DECIMAL(7,4). Example: '123.4567'.",
  "alien|sourceproperties|sourcedecdeg": "Full name: 'Declination (°)'. Explanation: Declination of the source in degrees (-90 to +90). Data type: DECIMAL(7,4). Example: '-20.4567'.",
  "alien|sourceproperties|sourcedistly": "Full name: 'Distance (ly)'. Explanation: Approximate distance to the source in light-years. Data type: NUMERIC(10,2). Example: '26000.45'.",
  "alien|sourceproperties|gallong": "Full name: 'Galactic Longitude (°)'. Explanation: Galactic longitude of the source. Data type: DECIMAL(6,2). Example: '12.34'.",
  "alien|sourceproperties|gallat": "Full name: 'Galactic Latitude (°)'. Explanation: Galactic latitude of the source. Data type: DECIMAL(6,2). Example: '-5.67'.",
  "alien|sourceproperties|celestobj": "Full name: 'Celestial Object'. Explanation: Broad classification of the object. Data type: VARCHAR(75). Possible categories: Galaxy, Planet, Star, Unknown.",
  "alien|sourceproperties|objtype": "Full name: 'Object Subtype'. Explanation: More specific object type. Data type: VARCHAR(50). Possible categories: Dwarf, Giant, Main Sequence, Unknown.",
  "alien|sourceproperties|objmag": "Full name: 'Apparent Magnitude'. Explanation: Brightness of the object as seen from Earth. Data type: NUMERIC(5,2). Example: '7.35'.",
  "alien|sourceproperties|objtempk": "Full name: 'Object Temperature (K)'. Explanation: Approximate surface temperature in Kelvin. Data type: INTEGER. Example: '5800'.",
  "alien|sourceproperties|objmasssol": "Full name: 'Object Mass (solar)'. Explanation: Mass relative to the Sun. Data type: DECIMAL(6,3). Example: '1.005'.",
  "alien|sourceproperties|objagegyr": "Full name: 'Object Age (Gyr)'. Explanation: Estimated age in billions of years. Data type: DECIMAL(6,3). Example: '4.500'.",
  "alien|sourceproperties|objmetal": "Full name: 'Metallicity'. Explanation: Ratio of elements heavier than helium in the object. Data type: NUMERIC(5,3). Example: '0.012'.",
  "alien|sourceproperties|objpropmotion": "Full name: 'Proper Motion (mas/yr)'. Explanation: Apparent motion across the sky in milliarcseconds/year. Data type: DECIMAL(7,2). Example: '55.12'.",
  "alien|sourceproperties|objradvel": "Full name: 'Radial Velocity (km/s)'. Explanation: Speed at which the object is moving toward/away from us. Data type: DECIMAL(7,2). Example: '-23.45'."
}

# External Knowledge:
[
  {
    "id": 0,
    "knowledge": "Signal-to-Noise Quality Indicator (SNQI)",
    "description": "Combines SNR and noise floor to provide a unified signal quality metric.",
    "definition": "$\\text{SNQI} = \\text{SnrRatio} - 0.1 \\times |\\text{NoiseFloorDbm}|$, where higher values indicate better detection quality. Positive values generally indicate analyzable signals."
  },
  {
    "id": 1,
    "knowledge": "Atmospheric Observability Index (AOI)",
    "description": "Quantifies how conducive atmospheric conditions are for signal detection.",
    "definition": "$\\text{AOI} = \\text{AtmosTransparency} \\times (1 - \\frac{\\text{HumidityRate}}{100}) \\times (1 - 0.02 \\times \\text{WindSpeedMs})$, where values closer to 1 indicate ideal observation conditions."
  },
  {
    "id": 2,
    "knowledge": "Signal Complexity Ratio (SCR)",
    "description": "Measures the relationship between signal complexity and information density.",
    "definition": "$\\text{SCR} = \\frac{\\text{ComplexIdx} \\times \\text{InfoDense}}{\\log(\\text{BwHz})}$, where higher values suggest potential artificial origin rather than natural phenomena."
  },
  {
    "id": 3,
    "knowledge": "Technological Origin Likelihood Score (TOLS)",
    "description": "Combines multiple factors to estimate likelihood of technological origin.",
    "definition": "$\\text{TOLS} = \\text{TechSigProb} \\times (1 - \\text{NatSrcProb}) \\times \\text{SigUnique} \\times (0.5 + \\frac{\\text{AnomScore}}{10})$, where values above 0.75 warrant further investigation as potential technosignatures."
  },
  {
    "id": 4,
    "knowledge": "Bandwidth-Frequency Ratio (BFR)",
    "description": "Measures the proportion of bandwidth to center frequency, helping identify signal type.",
    "definition": "$\\text{BFR} = \\frac{\\text{BwHz}}{\\text{CenterFreqMhz} \\times 10^6}$, where narrow ratios ($<0.001$) often indicate technological signals while wider ratios suggest natural phenomena."
  },
  {
    "id": 5,
    "knowledge": "Detection Instrument Sensitivity Factor (DISF)",
    "description": "Calculates the effective sensitivity of the detection setup based on telescope and environmental factors.",
    "definition": "$\\text{DISF} = (10 - \\frac{|\\text{AirTempC} - 15|}{10}) \\times \\text{AtmosTransparency} \\times (1 - \\frac{\\text{HumidityRate}}{200}) \\times \\frac{100 - \\text{LunarDistDeg}}{100}$, where values closer to 10 indicate optimal detection sensitivity."
  },
  {
    "id": 6,
    "knowledge": "Encoding Complexity Index (ECI)",
    "description": "Evaluates the sophistication of potential encoding in the signal.",
    "definition": "$\\text{ECI} = \\frac{\\text{CompressRatio} \\times \\text{ComplexIdx} \\times \\text{EntropyVal}}{10}$, where values above 1.5 suggest deliberate information encoding rather than random patterns."
  },
  {
    "id": 7,
    "knowledge": "Signal Stability Metric (SSM)",
    "description": "Quantifies overall temporal and spectral stability of a signal.",
    "definition": "$\\text{SSM} = (1 - \\frac{|\\text{FreqDriftHzs}|}{\\text{FreqMhz} \\times 1000}) \\times \\frac{\\text{SigDurSec}}{1 + \\frac{\\text{DoppShiftHz}}{1000}}$, where higher values indicate more stable signals typical of fixed transmitters."
  },
  {
    "id": 8,
    "knowledge": "Research Priority Index (RPI)",
    "description": "Helps researchers prioritize signals for follow-up based on multiple factors.",
    "definition": "$\\text{RPI} = (\\text{TechSigProb} \\times 4 + \\frac{\\text{BioSigProb}}{100} + \\text{SigUnique} \\times 2 + \\frac{\\text{AnomScore}}{2}) \\times (1 - \\text{FalsePosProb})$, where values above 3 indicate high research priority."
  },
  {
    "id": 9,
    "knowledge": "Lunar Interference Factor (LIF)",
    "description": "Calculates the potential interference from lunar illumination on observations.",
    "definition": "$\\text{LIF} = (1 - \\frac{\\text{LunarDistDeg}}{180}) \\times (1 - \\text{AtmosTransparency})$, where higher values indicate more lunar interference. Values above 0.5 suggest significant lunar contamination in data."
  },
  {
    "id": 10,
    "knowledge": "Technosignature",
    "description": "Defines the concept of signals that indicate technological activity.",
    "definition": "A signal with $\\text{TechSigProb} > 0.7$, $\\text{NatSrcProb} < 0.3$, and $\\text{ArtSrcProb} < 50$ that exhibits narrow bandwidth ($\\text{BFR} < 0.001$) and high information density ($\\text{InfoDense} > 0.8$)."
  },
  {
    "id": 11,
    "knowledge": "Coherent Information Pattern (CIP)",
    "description": "Identifies signals showing patterns consistent with deliberate information transmission.",
    "definition": "Signals characterized by high signal stability ($\\text{SSM} > 0.8$), organized information structure ($\\text{EntropyVal}$ between 0.4-0.8), and consistent modulation ($\\text{ModType}$ with $\\text{ModIndex} > 0.5$)."
  },
  {
    "id": 12,
    "knowledge": "Target of Opportunity (TOO)",
    "description": "Identifies high-value signals requiring immediate follow-up observation.",
    "definition": "Any signal with $\\text{RPI} > 3.5$, $\\text{TechSigProb} > 0.8$, and $\\text{AnomScore} > 5$ that has not been previously documented or explained by known phenomena."
  },
  {
    "id": 13,
    "knowledge": "Optimal Observing Window (OOW)",
    "description": "Defines conditions when observational quality is maximized.",
    "definition": "Time periods when $\\text{AOI} > 0.85$, $\\text{LunarStage}$ is 'New' or 'First Quarter', $\\text{LunarDistDeg} > 45$, and $\\text{SolarStatus}$ is 'Low' or 'Moderate'."
  },
  {
    "id": 14,
    "knowledge": "Signal Degradation Scenario (SDS)",
    "description": "Characterizes situations where signal quality is compromised by environmental factors.",
    "definition": "Observation conditions where one or more of: $\\text{AtmosTransparency} < 0.7$, $\\text{HumidityRate} > 70$, $\\text{WindSpeedMs} > 8$, or $\\text{GeomagStatus}$ contains 'Storm', resulting in compromised data quality."
  },
  {
    "id": 15,
    "knowledge": "Narrowband Technological Marker (NTM)",
    "description": "Identifies a specific signature associated with technological transmission.",
    "definition": "Signals with extremely narrow bandwidth ($\\text{BFR} < 0.0001$), stable frequency ($\\text{FreqDriftHzs} < 0.1$)."
  },
  {
    "id": 16,
    "knowledge": "Observational Confidence Level (OCL)",
    "description": "Rates the reliability of observations based on conditions and equipment.",
    "definition": "A classification system with three tiers: 'High' ($\\text{AOI} > 0.8$, $\\text{EquipStatus} = \\text{'Operational'}$, $\\text{CalibrStatus} = \\text{'Current'}$), 'Medium' ($\\text{AOI}$ 0.5-0.8, minor equipment issues), and 'Low' ($\\text{AOI} < 0.5$ or significant equipment problems)."
  },
  {
    "id": 17,
    "knowledge": "Potential Biosignature",
    "description": "Defines characteristics of signals potentially associated with biological processes.",
    "definition": "Signals with $\\text{BioSigProb} > 0.6$, $\\text{TechSigProb} < 0.4$, and spectral features that match known biological emission patterns, often associated with specific molecular transitions."
  },
  {
    "id": 18,
    "knowledge": "Encoded Information Transfer (EIT)",
    "description": "Characterizes signals that appear to contain deliberate information encoding.",
    "definition": "Signals with $\\text{ECI} > 1.8$, $\\text{EntropyVal}$ between 0.3-0.7 (not random but structured), and consistent internal patterns that suggest language or data encoding schemes."
  },
  {
    "id": 19,
    "knowledge": "Fast Radio Transient (FRT)",
    "description": "Defines a specific class of brief, high-energy radio emissions.",
    "definition": "Signals with extremely short duration ($\\text{SigDurSec} < 0.1$), high signal strength ($\\text{SigStrDb} > 15$), broad bandwidth ($\\text{BwHz} > 1000000$), and no periodicity ($\\text{RepeatCount} = 1$)."
  },
  {
    "id": 20,
    "knowledge": "WeathProfile: Clear",
    "description": "Illustrates optimal weather conditions for signal detection.",
    "definition": "Indicates pristine sky conditions with no clouds, usually associated with $\\text{AtmosTransparency} > 0.9$, low $\\text{HumidityRate} (< 40\\%)$, and minimal $\\text{WindSpeedMs} (< 3.0)$. Provides ideal visibility for optical observations and minimal atmospheric interference for radio observations."
  },
  {
    "id": 21,
    "knowledge": "SeeingProfile: Excellent",
    "description": "Illustrates superior atmospheric seeing conditions.",
    "definition": "Describes atmospheric conditions with minimal turbulence, allowing for sharp, detailed observations. Typically corresponds to image stability better than 1 arcsecond and is often associated with stable temperature gradients and low wind speeds ($\\text{WindSpeedMs} < 2.5$)."
  },
  {
    "id": 22,
    "knowledge": "SignalClass: Narrowband",
    "description": "Illustrates characteristics of narrowband signal detections.",
    "definition": "Describes signals occupying a very narrow portion of the spectrum (typically $\\text{BFR} < 0.0001$). Often associated with technological origins as natural sources rarely produce such spectrally confined emissions. These signals are particularly interesting in SETI research."
  },
  {
    "id": 23,
    "knowledge": "GeomagStatus: Major Storm",
    "description": "Illustrates severe geomagnetic disturbance conditions.",
    "definition": "Indicates intense solar-induced geomagnetic activity with Kp index ≥ 7. During such conditions, ionospheric perturbations significantly affect radio observations below 100 MHz, aurora may be visible at mid-latitudes, and satellite communications may experience disruptions."
  },
  {
    "id": 24,
    "knowledge": "CIP Classification Label",
    "description": "Three-tier rating system for evaluating signal coherence against intelligent transmission criteria.",
    "definition": "Classification labels: 'Coherent Information Pattern Detected' ($\\text{SSM} > 0.8$, $\\text{EntropyVal}$ between 0.4-0.8, and $\\text{ModIndex} > 0.5$), 'Potential Information Pattern' ($\\text{SSM} > 0.6$ and $\\text{EntropyVal}$ between 0.3-0.9$), or 'No Clear Pattern' (all other signals)."
  },
  {
    "id": 25,
    "knowledge": "SigClassType: Broadband Transient",
    "description": "Illustrates a class of brief signals covering wide frequency ranges.",
    "definition": "Describes short-duration signals ($\\text{SigDurSec}$ typically $< 5$) that span a large portion of the spectrum ($\\text{BFR} > 0.1$). Examples include solar radio bursts, lightning discharges, and certain types of cosmic explosions like Fast Radio Bursts (FRBs)."
  },
  {
    "id": 26,
    "knowledge": "PolarMode: Circular",
    "description": "Illustrates circular polarization in detected signals.",
    "definition": "Describes electromagnetic waves where the electric field vector rotates in a circular pattern as the wave propagates. Circular polarization maintaining high purity across frequency (indicated by $\\text{PolarAngleDeg}$ stability) is rare in natural sources and may indicate technological origin."
  },
  {
    "id": 27,
    "knowledge": "EncryptEvid: Strong Pattern",
    "description": "Illustrates characteristics suggesting deliberate signal encoding.",
    "definition": "Indicates detection of non-random, internally consistent patterns that resist simple decoding but show hallmarks of designed encryption or encoding. Characterized by high $\\text{EntropyVal} (> 0.7)$ combined with structural regularity that defies natural explanation."
  },
  {
    "id": 28,
    "knowledge": "EncodeType: Frequency Hopping",
    "description": "Illustrates a sophisticated encoding method used in telecommunications.",
    "definition": "Describes a transmission technique where the signal rapidly switches frequencies according to a predetermined sequence. Detection would be characterized by discontinuous spectral features that follow a pattern. This technique is used on Earth to secure communications and reduce interference."
  },
  {
    "id": 29,
    "knowledge": "FalsePosProb: <0.01",
    "description": "Illustrates extremely high confidence in signal detection.",
    "definition": "Indicates less than 1% probability that the signal is a false detection or artifact. Such low false positive probability typically results from multiple independent confirmations, excellent signal strength (high $\\text{SnrRatio}$), and elimination of all known terrestrial and instrumental sources."
  },
  {
    "id": 30,
    "knowledge": "Modulation Complexity Score (MCS)",
    "description": "Quantifies the sophistication of signal modulation based on type and stability.",
    "definition": "$\\text{MCS} = \\text{ModIndex} \\times (1 + \\text{SSM}) \\times M_{\\text{factor}}$, where $M_{\\text{factor}}$ is 2 for $\\text{ModType} = \\text{'AM'}$, 1.5 for 'FM', and 1 for other types. Incorporates Signal Stability Metric (SSM) to weight stable modulations higher."
  },
  {
    "id": 31,
    "knowledge": "Artificial Intelligence Detection Probability (AIDP)",
    "description": "Calculates likelihood of artificial intelligence origin based on encoding complexity and technosignature indicators.",
    "definition": "$\\text{AIDP} = \\frac{\\text{ECI} \\times \\text{TOLS}}{1 + \\text{NatSrcProb}}$, where ECI (Encoding Complexity Index) and TOLS (Technological Origin Likelihood Score) are weighted against natural source probability."
  },
  {
    "id": 32,
    "knowledge": "Observation Quality Factor (OQF)",
    "description": "Provides a comprehensive measure of observational conditions quality.",
    "definition": "$\\text{OQF} = \\text{AOI} \\times (1 - \\text{LIF}) \\times (\\text{PointAccArc} < 2 ? 1 : \\frac{2}{\\text{PointAccArc}})$, where AOI (Atmospheric Observability Index) and LIF (Lunar Interference Factor) are combined with telescope pointing accuracy."
  },
  {
    "id": 33,
    "knowledge": "Information Entropy Ratio (IER)",
    "description": "Compares signal entropy to expected natural background entropy.",
    "definition": "$\\text{IER} = \\frac{\\text{EntropyVal}}{\\text{NatSrcProb} \\times 0.9 + 0.1}$, where values significantly greater than 1 suggest non-natural information content. Uses NatSrcProb as a baseline for expected natural entropy."
  },
  {
    "id": 34,
    "knowledge": "Signal Processing Efficiency Index (SPEI)",
    "description": "Evaluates the computational efficiency of signal processing relative to complexity.",
    "definition": "$\\text{SPEI} = \\frac{\\text{DecodeIters} \\times \\text{ProcTimeHrs}}{\\text{ECI} \\times \\text{ComplexIdx}}$, where ECI (Encoding Complexity Index) provides the complexity component to normalize processing time and iterations."
  },
  {
    "id": 35,
    "knowledge": "Celestial Location Significance Factor (CLSF)",
    "description": "Calculates significance of signal source location based on astronomical targets of interest.",
    "definition": "$\\text{CLSF} = (\\text{CelestObj} ? 2 : 1) \\times (\\text{ObjType} == \\text{'Giant'} \\&\\& \\text{ObjMassSol} \\text{ between } 0.8 \\text{ and } 1.2 ? 1.5 : 1) \\times (\\text{ObjMetal} > 0 ? \\text{ObjMetal} + 1 : 0.5)$, where higher values indicate source locations more likely to harbor intelligent life."
  },
  {
    "id": 36,
    "knowledge": "Confirmation Confidence Score (CCS)",
    "description": "Quantifies overall confidence in signal verification across multiple parameters.",
    "definition": "$\\text{CCS} = (1 - \\text{FalsePosProb}) \\times \\text{DecodeConf} \\times \\text{ClassConf} \\times (\\text{SNQI} > 0 ? \\frac{\\text{SNQI}}{10} + 0.5 : 0.1)$, where SNQI (Signal-to-Noise Quality Indicator) provides a quality weighting factor."
  },
  {
    "id": 37,
    "knowledge": "Habitable Zone Signal Relevance (HZSR)",
    "description": "Assesses signal relevance based on source's position in habitable zone.",
    "definition": "$\\text{HZSR} = \\text{TOLS} \\times (\\text{ObjType} == \\text{'Dwarf'} ? (0.7 \\leq \\text{ObjMassSol} \\leq 1.4 ? (0.8 \\leq \\frac{\\text{SourceDistLy}}{\\sqrt{\\text{ObjMassSol}}} \\leq 1.7 ? 2 : 0.5) : 0.3) : 0.1)$, where TOLS (Technological Origin Likelihood Score) is weighted by stellar habitability factors."
  },
  {
    "id": 38,
    "knowledge": "Pattern Recognition Confidence (PRC)",
    "description": "Measures confidence in identified signal patterns based on multiple factors.",
    "definition": "$\\text{PRC} = (\\text{RepeatCount} > 1 ? 1 + \\log_{10}(\\text{RepeatCount}) : 0.5) \\times (\\text{EntropyVal} < 0.9 ? 1 : 0.3) \\times \\text{SCR}$, where SCR (Signal Complexity Ratio) provides complexity weighting."
  },
  {
    "id": 39,
    "knowledge": "NTM Classification System",
    "description": "A tiered classification system for Narrowband Technological Markers based on signal characteristics.",
    "definition": "Three-tier classification: 'Strong NTM' (BFR < 0.0001 AND FreqDriftHzs < 0.1 AND non-natural modulation), 'Moderate NTM' (BFR < 0.0005 AND FreqDriftHzs < 0.5 AND non-natural modulation), and 'Not NTM' (all other signals)."
  },
  {
    "id": 40,
    "knowledge": "High-Confidence Technosignature",
    "description": "Defines signals with extremely high likelihood of technological origin.",
    "definition": "A Technosignature with $\\text{CCS} > 0.9$, $\\text{MCS} > 1.5$, and $\\text{AIDP} > 0.8$, indicating a signal that meets the basic Technosignature criteria with additional confirmation through modulation complexity and artificial intelligence detection markers."
  },
  {
    "id": 41,
    "knowledge": "Habitable Zone Transmission",
    "description": "Identifies signals originating from stellar habitable zones with technological characteristics.",
    "definition": "A signal with $\\text{HZSR} > 1.5$ and Technosignature characteristics, originating from a star system with conditions potentially suitable for life, making it a priority candidate for SETI research."
  },
  {
    "id": 42,
    "knowledge": "Multi-Channel Communication Protocol",
    "description": "Identifies signal patterns consistent with sophisticated communication protocols.",
    "definition": "Signal exhibiting Coherent Information Pattern (CIP) characteristics across multiple frequency channels with coordinated timing ($\\text{RepeatCount} > 3$, $\\text{PeriodSec}$ consistent across observations) and $\\text{ECI} > 2.0$, suggesting a designed communication system."
  },
  {
    "id": 43,
    "knowledge": "Quantum-Coherent Transmission",
    "description": "Describes signals potentially employing quantum properties for communication.",
    "definition": "Signals with $\\text{QuantEffects}$ containing 'Significant' or 'Observed' patterns, exhibiting unusually high information density ($\\text{InfoDense} > 1.5$) while maintaining an $\\text{ECI} > 2.5$, suggesting advanced transmission technologies beyond conventional radiofrequency methods."
  },
  {
    "id": 44,
    "knowledge": "Research Critical Signal",
    "description": "Defines signals requiring immediate and extensive scientific resources.",
    "definition": "Signals meeting Target of Opportunity (TOO) criteria with additional $\\text{PRC} > 0.8$ and $\\text{IMDF} < 0.5$, indicating high-quality, minimally distorted signals that show recognizable patterns warranting priority allocation of research resources."
  },
  {
    "id": 45,
    "knowledge": "Directed Transmission",
    "description": "Identifies signals that appear specifically directed rather than omnidirectional.",
    "definition": "Signals with high spatial stability ($\\text{SpatStab} = \\text{'Moderate'}$), narrow beam characteristics ($\\text{PolarMode} = \\text{'Linear'}$ with stable $\\text{PolarAngleDeg}$), and high $\\text{TOLS} > 0.85$, suggesting intentional transmission toward our location."
  },
  {
    "id": 46,
    "knowledge": "Signal of Galactic Significance",
    "description": "Classifies signals with potential importance to galactic civilization models.",
    "definition": "Signals originating from regions of high $\\text{CLSF} (> 2.0)$ that display Technosignature characteristics and have $\\text{AIDP} > 0.7$, representing potential evidence of advanced civilizations at galactic-relevant locations."
  },
  {
    "id": 47,
    "knowledge": "CCS Approximation",
    "description": "Simplified CCS calculation using direct signal-to-noise ratio values when full Signal-to-Noise Quality Indicator (SNQI) data is unavailable.",
    "definition": "$(1 - \\text{FalsePosProb}) \\times \\text{DecodeConf} \\times (\\text{SNR} - 0.1 \\times |\\text{NoiseFloorDbm}| > 0 ? \\frac{\\text{SNR} - 0.1 \\times |\\text{NoiseFloorDbm}|}{10} + 0.5 : 0.1)$"
  },
  {
    "id": 48,
    "knowledge": "Observation-Verified Signal",
    "description": "Defines signals that have undergone rigorous verification processes.",
    "definition": "Signals observed under Optimal Observing Window (OOW) conditions with $\\text{OQF} > 0.85$ and $\\text{CCS} > 0.8$, indicating high-quality observations with multiple verification methods applied."
  },
  {
    "id": 49,
    "knowledge": "Anomalous Quantum Signal",
    "description": "Describes signals exhibiting quantum properties inconsistent with current physics models.",
    "definition": "Signals with $\\text{QuantEffects}$ indicating anomalous behavior, $\\text{AnomScore} > 8$, and unusually high $\\text{MCS} (> 2.0)$, suggesting either unknown natural quantum phenomena or extremely advanced transmission technologies beyond current human capabilities."
  },
  {
    "id": 50,
    "knowledge": "Analyzable Signals",
    "description": "Signals of sufficient quality to be considered useful for further analysis.",
    "definition": "Signals with SNQI > 0 are considered analyzable."
  },
  {
    "id": 51,
    "knowledge": "Bandwidth-to-Frequency Ratio (BFR)",
    "description": "Normalized signal width relative to its central frequency.",
    "definition": "$\\text{BFR} = \\frac{\\text{BwHz}}{\\text{CenterFreqMhz} \\times 1{,}000{,}000}$, used to characterize signal spread relative to its frequency band."
  },
  {
    "id": 52,
    "knowledge": "TOLS Category",
    "description": "Classification of signals based on TOLS thresholds.",
    "definition": "Categorized as 'Low' if TOLS < 0.25, 'Medium' if TOLS < 0.75, and 'High' otherwise."
  },
  {
    "id": 53,
    "knowledge": "High Lunar Interference Events",
    "description": "Observations with significant lunar interference.",
    "definition": "Events where the calculated LIF is greater than 0.5, indicating strong lunar contamination in the data."
  },
  {
    "id": 54,
    "knowledge": "High Confidence Signals",
    "description": "Signal with Confirmation Confidence Score (CCS) > 0.8, indicating high reliability.",
    "definition": "Signals where $\\text{CCS} > 0.8$"
  },
  {
    "id": 55,
    "knowledge": "Equipment Problems",
    "description": "Defines what counts as an abnormal condition for a telescope’s subsystems.",
    "definition": "A telescope is considered to have an equipment problem whenever **any** of its key subsystem states are not in their nominal condition: • Equipment status is not “Operational”; • Calibration status is not “Current”; • Cooling-system status is not “Normal”."
  }
]

# User Task:
I want to analyze how the Signal-to-Noise Quality Indicator (SNQI) varies across different weather conditions. For each weather condition, give weather condition name, the average SNQI, the median SNQI, and count how many analyzable signals there are. Sort the result by average SNQI in descending order.

Generate the correct PostgreSQL to handle the user task above:
(FORMAT: You should enclose your final PostgreSQL in '```postgresql
[Your Generated SQLs]
```' in the end. Could use semicolon to separate multiple statements.)

# Your Generated SQL:
```postgresql
````

#### Role description

Overall management and access to all system tables.

#### Gold SQL

````
WITH signal_quality AS ( SELECT s.SignalRegistry, s.SnrRatio - 0.1 * ABS(s.NoiseFloorDbm) AS SNQI, o.WeathProfile FROM Signals s JOIN Telescopes t ON s.TelescRef = t.TelescRegistry JOIN Observatories o ON t.ObservStation = o.ObservStation ), ranked_quality AS ( SELECT WeathProfile, SNQI, ROW_NUMBER() OVER (PARTITION BY WeathProfile ORDER BY SNQI) as row_num, COUNT(*) OVER (PARTITION BY WeathProfile) as total_count FROM signal_quality ) SELECT WeathProfile, AVG(SNQI) AS avg_snqi, (SELECT SNQI FROM ranked_quality WHERE WeathProfile = rq.WeathProfile AND row_num = (total_count + 1) / 2) AS median_snqi, SUM(CASE WHEN SNQI > 0 THEN 1 ELSE 0 END) AS analyzable_signals FROM ranked_quality rq GROUP BY WeathProfile ORDER BY avg_snqi DESC;
````

#### External knowledge

````
[0, 50]
````

### 下一步操作提示
- 若需引入 UPDATE/DELETE 等语句，请在 `allowed_statement_types` 中放开对应类型并确保角色权限模型已扩展。
- 生成的完整 prompt 字段参考官方 LiveSQLBench 模板，如需自定义可扩展 `LiveSQLRoleSQLGenerator`。
- 导出文件位于 `outputs/livesql_data/`，可结合评测脚本或下游任务继续处理。

In [14]:
baseline_diagnostics

{'total_rows': 872,
 'missing_instance_id': 0,
 'duplicate_instances': 694,
 'missing_gold_sql': 0,
 'unique_instances': 178}

In [15]:
sql_generator.get_last_statement_type_counts(), sql_generator.get_last_total_example_count(), sql_generator.get_last_filtered_example_count()

({'select': 178, 'write': 64, 'ddl': 27, 'unknown': 1}, 270, 92)